# Synthetic Wound Progression Generator
**Generate temporal healing/worsening sequences using Stable Diffusion**

This notebook creates synthetic wound progression images from real baseline images.

**Run this in Google Colab with GPU runtime!**

## Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install diffusers and dependencies
!pip install -q diffusers transformers accelerate safetensors pillow torch torchvision

In [ ]:
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Load Stable Diffusion Model

Using Stable Diffusion 2.1 for img2img generation

In [ ]:
model_id = "stabilityai/stable-diffusion-2-1"

print("Loading Stable Diffusion pipeline...")
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None  # Disable for medical images
)
pipe = pipe.to(device)

print("✓ Pipeline loaded")

## Upload Baseline Wound Images

Upload 30-50 wound images from your datasets to use as baselines

In [ ]:
from google.colab import files

print("Upload baseline wound images (Day 0):")
uploaded = files.upload()

baseline_images = list(uploaded.keys())
print(f"\nUploaded {len(baseline_images)} baseline images")

## Define Progression Prompts

Medical-specific prompts for healing and worsening scenarios

In [ ]:
# Healing progression prompts (improving)
HEALING_PROMPTS = {
    "day_7": "medical wound photo, 15% smaller wound, increased pink granulation tissue, reduced exudate, healthier wound bed, clinical photography",
    "day_14": "medical wound photo, 35% smaller wound, epithelial tissue forming at edges, pink healthy granulation, reduced inflammation, clinical photography",
    "day_21": "medical wound photo, 60% smaller wound, re-epithelialization progressing, healthy pink tissue, minimal exudate, healing margins, clinical photography"
}

# Worsening progression prompts (deteriorating)
WORSENING_PROMPTS = {
    "day_7": "medical wound photo, 10% larger wound, increased slough tissue, yellow exudate, inflamed surrounding skin, clinical photography",
    "day_14": "medical wound photo, 25% larger wound, necrotic tissue present, purulent exudate, macerated edges, clinical photography",
    "day_21": "medical wound photo, 40% larger wound, significant necrosis, heavy purulent drainage, erythematous surrounding tissue, clinical photography"
}

# Negative prompt (what to avoid)
NEGATIVE_PROMPT = "cartoon, drawing, illustration, text, watermark, low quality, blurry, distorted, unrealistic"

## Generate Synthetic Progressions

In [ ]:
def generate_wound_progression(baseline_path, wound_id, progression_type="healing"):
    """
    Generate temporal sequence for a wound
    
    Args:
        baseline_path: Path to Day 0 image
        wound_id: Unique identifier
        progression_type: 'healing' or 'worsening'
    """
    
    # Load baseline image
    baseline_img = Image.open(baseline_path).convert('RGB')
    baseline_img = baseline_img.resize((512, 512))
    
    # Select prompts
    prompts = HEALING_PROMPTS if progression_type == "healing" else WORSENING_PROMPTS
    
    sequence = {"day_0": baseline_img}
    
    for timepoint, prompt in prompts.items():
        print(f"  Generating {timepoint}...")
        
        # Generate progression image
        result = pipe(
            prompt=prompt,
            image=baseline_img,
            strength=0.3,  # Keep 70% of original structure
            guidance_scale=7.5,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=30
        ).images[0]
        
        sequence[timepoint] = result
    
    return sequence

# Save sequence
def save_sequence(sequence, wound_id, output_dir="synthetic_wounds"):
    """Save generated sequence to files"""
    output_path = Path(output_dir) / wound_id
    output_path.mkdir(parents=True, exist_ok=True)
    
    for timepoint, img in sequence.items():
        img.save(output_path / f"{timepoint}.png")
    
    print(f"✓ Saved sequence to {output_path}")

## Batch Generate All Sequences

In [ ]:
# Generate healing sequences (80% of baselines)
num_healing = int(len(baseline_images) * 0.8)
num_worsening = len(baseline_images) - num_healing

print(f"Generating {num_healing} healing sequences and {num_worsening} worsening sequences...\n")

for idx, baseline_path in enumerate(baseline_images):
    wound_id = f"wound_{idx:03d}"
    progression = "healing" if idx < num_healing else "worsening"
    
    print(f"[{idx+1}/{len(baseline_images)}] {wound_id} ({progression})")
    
    sequence = generate_wound_progression(baseline_path, wound_id, progression)
    save_sequence(sequence, wound_id)
    
    # Visualize first sequence
    if idx == 0:
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        for ax, (timepoint, img) in zip(axes, sequence.items()):
            ax.imshow(img)
            ax.set_title(timepoint.replace('_', ' ').title())
            ax.axis('off')
        plt.tight_layout()
        plt.show()

print("\n✅ All sequences generated!")

## Download Generated Sequences

In [ ]:
# Zip all synthetic wounds
!zip -r synthetic_wounds.zip synthetic_wounds/

# Download
from google.colab import files
files.download('synthetic_wounds.zip')

print("✓ Download complete! Upload this to your Kaggle notebook.")

## Summary

You've now generated:
- **Baseline images** (Day 0)
- **Healing sequences** (~40 wounds × 4 timepoints)
- **Worsening sequences** (~10 wounds × 4 timepoints)

**Total**: ~200 images ready for WoundTrack analysis!

Upload `synthetic_wounds.zip` to your Kaggle notebook and continue with MedGemma analysis.